## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Repo root (parent of this notebook's folder)
NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC = REPO_ROOT / "src"

for p in (str(REPO_ROOT), str(SRC)):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch

import playground as pg

print("Repo root:", REPO_ROOT)
print("Device (for training):", "cuda" if torch.cuda.is_available() else "cpu")

## 2. Configuration

In [ ]:
DATA_DIR = str(REPO_ROOT / "data")

TRAIN_KWARGS = dict(
    data_dir=DATA_DIR,
    deduplicate=False,
    test_size=0.2,
    random_state=42,
    loss_plot_path=str(REPO_ROOT / "training_loss.png"),
    show_loss_plot=True,
    # EASY MODE SETTINGS
    epochs=1000,                 # how many times the model will see the entire training dataset
    batch_size=256,              # how many samples the model sees before updating its weights
    mlp_hidden_dims=(128, 64),   # number of neurons in each hidden layer of the MLP
    lr=1e-2,                     # learning rate for the optimizer (how big the steps are during training)
    # MEDIUM MODE SETTINGS
    early_stopping=True,         # whether to stop training early if the validation loss doesn't improve for a certain number of epochs
    early_stopping_patience=40,  # how many epochs to wait for improvement before stopping training
    early_stopping_min_delta=0.0,# minimum change in validation loss to qualify as an improvement (0.0 means any improvement will reset the patience counter
    # HARDCORE MODE SETTINGS
    mlp_dropouts=0.2,            # dropout rates for each hidden layer (None means no dropout)
    weight_decay=1e-2,           # L2 regularization strength (penalty for large weights to prevent overfitting)
    lr_decay=True,               # whether to reduce the learning rate if the validation loss plateaus
    lr_decay_factor=0.9,         # how stubborn will the learning rate be (0.8 means it will be reduced to 80% of its value when triggered)
    lr_decay_patience=15,        # how many epochs to wait for improvement before reducing the learning rate
)

print("DATA_DIR:", DATA_DIR)
print("JSONL files:", list(Path(DATA_DIR).rglob("*.jsonl"))[:10], "...")

## 3. Inspect data (optional)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

peek = pg.load_training_engineered_xy(DATA_DIR, deduplicate=TRAIN_KWARGS["deduplicate"])
X, y, sample_weights, sample_groups = peek
if X is None:
    raise FileNotFoundError(f"No rows under {DATA_DIR!r}. Record gameplay or fix DATA_DIR.")

print("X shape:", X.shape, "y shape:", y.shape)
print("Unique actions:", np.unique(y))
print("Action counts:", np.bincount(y))

splitter = GroupShuffleSplit(
    n_splits=1, test_size=TRAIN_KWARGS["test_size"], random_state=TRAIN_KWARGS["random_state"]
)
train_idx, test_idx = next(splitter.split(X, y, groups=sample_groups))
print("Train rows:", len(train_idx), "| Hold-out rows:", len(test_idx))
print("Unique sessions (train):", len(np.unique(sample_groups[train_idx])))
print("Unique sessions (hold-out):", len(np.unique(sample_groups[test_idx])))

## 4. Train the MLP

This calls the same routine as `python src/playground.py`: weighted cross-entropy, `ReduceLROnPlateau`, optional early stopping, best checkpoint by validation loss, then a train/val loss figure.

In [ ]:
model, device = pg.train_pytorch_classifier(**TRAIN_KWARGS)

## 5. Run the policy in Godot (optional)

Requires `src/.config` / `.env` pointing at your exported Godot build (same as `setup_environment()` in other scripts). If you only care about offline training, skip this cell.

In [ ]:
from utils import setup_environment

env = setup_environment()
obs = env.reset()
nb_agents = len(obs["obs"])
session_infos = [{"prev_action": 0} for _ in range(nb_agents)]

step_count = 0
while True:
    actions = [
        pg.agent_brain(obs["obs"][i], session_infos[i], step_count, model, device)
        for i in range(nb_agents)
    ]
    actions = np.array(actions, dtype=np.int64)

    for i in range(nb_agents):
        session_infos[i]["prev_action"] = int(np.asarray(actions[i]).flat[0])

    obs, reward, done, info = env.step(actions)
    if any(done):
        break
    step_count += 1

env.close()
print("Episode finished.")